In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql.window import *
from delta.tables import DeltaTable

In [0]:
df_old = spark.sql("SELECT * FROM db_catalog.silver.customers_silver")
df_old.display()

customer_id,email,city,state,full_name,domains,email_category
C00001,rushjeff@ryan.org,Johnsonmouth,MS,Emily Mooney,ryan.org,Other
C00002,mccoykiara@kelly.com,Stephenfort,WY,Andrea Sellers,kelly.com,Other
C00003,rebeccamiller@yahoo.com,South Stephenshire,LA,Craig Hayes,yahoo.com,Yahoo
C00004,lawrence05@campbell.info,Chrisland,ND,Bryan Scott,campbell.info,Other
C00005,carrie45@yahoo.com,East Dennistown,RI,Sean Vasquez,yahoo.com,Yahoo
C00006,traceyramos@gmail.com,North Matthew,IN,Kevin Mccarthy,gmail.com,Gmail
C00007,scottallen@gmail.com,Joneshaven,VA,Amanda Doyle,gmail.com,Gmail
C00008,sullivanjeremy@horton-adams.com,South Nathanfurt,CT,Paul Campos,horton-adams.com,Other
C00009,dennis03@yahoo.com,Kimberlyview,MD,Mary Green,yahoo.com,Yahoo
C00010,charles58@murillo.net,West Hector,OK,James Myers,murillo.net,Other


In [0]:
init_load_flag = not spark.catalog.tableExists(
    "db_catalog.gold.dim_customers"
)

if init_load_flag:

    df_dim_customers = (
        df_old
        .withColumn(
            "dim_customer_key",
            monotonically_increasing_id() + lit(1)
        )
        .withColumn(
            "create_date",
            current_timestamp()
        )
        .withColumn(
            "update_date",
            current_timestamp()
        )
        .withColumn(
            "current_flag",
            lit(True)
        )
    )

    df_dim_customers = df_dim_customers.select(
        "dim_customer_key",
        "customer_id",
        "email",
        "city",
        "state",
        "full_name",
        "domains",
        "email_category",
        "create_date",
        "update_date",
        "current_flag"
    )

    df_dim_customers.write \
        .format("delta") \
        .mode("overwrite") \
        .save(
            "abfss://gold@dbproject.dfs.core.windows.net/customers"
        )

    spark.sql("""
        CREATE TABLE IF NOT EXISTS db_catalog.gold.dim_customers
        USING DELTA
        LOCATION 'abfss://gold@dbproject.dfs.core.windows.net/customers'
    """)

else:
    df_customers_gold = spark.table(
        "db_catalog.gold.dim_customers"
    )
    max_dim_customer_key = (
        df_customers_gold
        .agg(
            max("dim_customer_key").alias("max_key")
        )
        .collect()[0]["max_key"]
    )

    max_dim_customer_key = max_dim_customer_key or 0

    df_customer_changes = (
        df_old.alias("source")
        .join(
            df_customers_gold.alias("gold"),
            col("source.customer_id") ==
            col("gold.customer_id"),
            "left"
        )
    )

    df_customer_changes = df_customer_changes.withColumn(
        "is_changed",
        (
            ~col("source.email").eqNullSafe(col("gold.email"))
        )
        |
        (
            ~col("source.city").eqNullSafe(col("gold.city"))
        )
        |
        (
            ~col("source.state").eqNullSafe(col("gold.state"))
        )
        |
        (
            ~col("source.full_name").eqNullSafe(col("gold.full_name"))
        )
        |
        (
            ~col("source.domains").eqNullSafe(col("gold.domains"))
        )
        |
        (
            ~col("source.email_category").eqNullSafe(
                col("gold.email_category")
            )
        )
    )

    df_customer_changes = df_customer_changes.withColumn(
        "is_new",
        col("gold.customer_id").isNull()
    )

    window_spec = Window.orderBy("source.customer_id")

    df_new_customers = (
        df_customer_changes
        .filter(col("is_new") == True)
        .withColumn(
            "dim_customer_key",
            row_number().over(window_spec)
            + max_dim_customer_key
        )
        .select(
            col("source.customer_id").alias("customer_id"),
            col("source.email").alias("email"),
            col("source.city").alias("city"),
            col("source.state").alias("state"),
            col("source.full_name").alias("full_name"),
            col("source.domains").alias("domains"),
            col("source.email_category").alias(
                "email_category"
            ),
            col("dim_customer_key")
        )
        .withColumn(
            "is_changed",
            lit(False)
        )
    )

    df_existing_customers = (
        df_customer_changes
        .filter(col("is_new") == False)
        .select(
            col("source.customer_id").alias("customer_id"),
            col("source.email").alias("email"),
            col("source.city").alias("city"),
            col("source.state").alias("state"),
            col("source.full_name").alias("full_name"),
            col("source.domains").alias("domains"),
            col("source.email_category").alias(
                "email_category"
            ),
            col("gold.dim_customer_key").alias(
                "dim_customer_key"
            ),
            col("is_changed")
        )
    )

    df_customer_merge = (
        df_existing_customers
        .unionByName(df_new_customers)
        .withColumn(
            "update_date",
            current_timestamp()
        )
    )

    delta_customers = DeltaTable.forName(
        spark,
        "db_catalog.gold.dim_customers"
    )

    (
        delta_customers.alias("gold")
        .merge(
            df_customer_merge.alias("source"),
            "gold.customer_id = source.customer_id"
        )
        .whenMatchedUpdate(
            condition="source.is_changed = true",
            set={
                "email": "source.email",
                "city": "source.city",
                "state": "source.state",
                "full_name": "source.full_name",
                "domains": "source.domains",
                "email_category": "source.email_category",
                "update_date": "source.update_date",
                "current_flag": "true"
            }
        )
        .whenNotMatchedInsert(
            values={
                "dim_customer_key":
                    "source.dim_customer_key",

                "customer_id":
                    "source.customer_id",

                "email":
                    "source.email",

                "city":
                    "source.city",

                "state":
                    "source.state",

                "full_name":
                    "source.full_name",

                "domains":
                    "source.domains",

                "email_category":
                    "source.email_category",

                "create_date":
                    "source.update_date",

                "update_date":
                    "source.update_date",

                "current_flag":
                    "true"
            }
        ).execute()
    )

    print("SCD Type 1 merge completed")

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


SCD Type 1 merge completed


/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


In [0]:
df_customers_gold = spark.table(
    "db_catalog.gold.dim_customers"
)

df_customers_gold.display()

dim_customer_key,customer_id,email,city,state,full_name,domains,email_category,create_date,update_date,current_flag
1,C00001,rushjeff@ryan.org,Johnsonmouth,MS,Emily Mooney,ryan.org,Other,2026-09-10T12:45:41.310Z,2026-09-10T12:45:41.310Z,true
2,C00002,mccoykiara@kelly.com,Stephenfort,WY,Andrea Sellers,kelly.com,Other,2026-09-10T12:45:41.310Z,2026-09-10T12:45:41.310Z,true
3,C00003,rebeccamiller@yahoo.com,South Stephenshire,LA,Craig Hayes,yahoo.com,Yahoo,2026-09-10T12:45:41.310Z,2026-09-10T12:45:41.310Z,true
4,C00004,lawrence05@campbell.info,Chrisland,ND,Bryan Scott,campbell.info,Other,2026-09-10T12:45:41.310Z,2026-09-10T12:45:41.310Z,true
5,C00005,carrie45@yahoo.com,East Dennistown,RI,Sean Vasquez,yahoo.com,Yahoo,2026-09-10T12:45:41.310Z,2026-09-10T12:45:41.310Z,true
6,C00006,traceyramos@gmail.com,North Matthew,IN,Kevin Mccarthy,gmail.com,Gmail,2026-09-10T12:45:41.310Z,2026-09-10T12:45:41.310Z,true
7,C00007,scottallen@gmail.com,Joneshaven,VA,Amanda Doyle,gmail.com,Gmail,2026-09-10T12:45:41.310Z,2026-09-10T12:45:41.310Z,true
8,C00008,sullivanjeremy@horton-adams.com,South Nathanfurt,CT,Paul Campos,horton-adams.com,Other,2026-09-10T12:45:41.310Z,2026-09-10T12:45:41.310Z,true
9,C00009,dennis03@yahoo.com,Kimberlyview,MD,Mary Green,yahoo.com,Yahoo,2026-09-10T12:45:41.310Z,2026-09-10T12:45:41.310Z,true
10,C00010,charles58@murillo.net,West Hector,OK,James Myers,murillo.net,Other,2026-09-10T12:45:41.310Z,2026-09-10T12:45:41.310Z,true


In [0]:
spark.sql("""
DESCRIBE DETAIL db_catalog.gold.dim_customers
""").display()

format,id,name,description,location,createdAt,lastModified,partitionColumns,clusteringColumns,numFiles,sizeInBytes,properties,minReaderVersion,minWriterVersion,tableFeatures,statistics,clusterByAuto
delta,13cfe0b7-0b48-4389-b3b1-d884eeebaee5,db_catalog.gold.dim_customers,null,abfss://gold@dbproject.dfs.core.windows.net/customers,2026-09-10T12:45:40.380Z,2026-09-10T14:31:16.000Z,List(),List(),1,94786,Map(delta.enableDeletionVectors -> true),3,7,"List(appendOnly, deletionVectors, invariants)","Map(numRowsDeletedByDeletionVectors -> 0, numDeletionVectors -> 0)",false


In [0]:
df = spark.read.format("delta").load("abfss://gold@dbproject.dfs.core.windows.net/customers")
df.display()

dim_customer_key,customer_id,email,city,state,full_name,domains,email_category,create_date,update_date,current_flag
1,C00001,rushjeff@ryan.org,Johnsonmouth,MS,Emily Mooney,ryan.org,Other,2026-09-10T12:45:41.310Z,2026-09-10T12:45:41.310Z,true
2,C00002,mccoykiara@kelly.com,Stephenfort,WY,Andrea Sellers,kelly.com,Other,2026-09-10T12:45:41.310Z,2026-09-10T12:45:41.310Z,true
3,C00003,rebeccamiller@yahoo.com,South Stephenshire,LA,Craig Hayes,yahoo.com,Yahoo,2026-09-10T12:45:41.310Z,2026-09-10T12:45:41.310Z,true
4,C00004,lawrence05@campbell.info,Chrisland,ND,Bryan Scott,campbell.info,Other,2026-09-10T12:45:41.310Z,2026-09-10T12:45:41.310Z,true
5,C00005,carrie45@yahoo.com,East Dennistown,RI,Sean Vasquez,yahoo.com,Yahoo,2026-09-10T12:45:41.310Z,2026-09-10T12:45:41.310Z,true
6,C00006,traceyramos@gmail.com,North Matthew,IN,Kevin Mccarthy,gmail.com,Gmail,2026-09-10T12:45:41.310Z,2026-09-10T12:45:41.310Z,true
7,C00007,scottallen@gmail.com,Joneshaven,VA,Amanda Doyle,gmail.com,Gmail,2026-09-10T12:45:41.310Z,2026-09-10T12:45:41.310Z,true
8,C00008,sullivanjeremy@horton-adams.com,South Nathanfurt,CT,Paul Campos,horton-adams.com,Other,2026-09-10T12:45:41.310Z,2026-09-10T12:45:41.310Z,true
9,C00009,dennis03@yahoo.com,Kimberlyview,MD,Mary Green,yahoo.com,Yahoo,2026-09-10T12:45:41.310Z,2026-09-10T12:45:41.310Z,true
10,C00010,charles58@murillo.net,West Hector,OK,James Myers,murillo.net,Other,2026-09-10T12:45:41.310Z,2026-09-10T12:45:41.310Z,true


In [0]:
%sql
select * from db_catalog.gold.dim_customers;

dim_customer_key,customer_id,email,city,state,full_name,domains,email_category,create_date,update_date,current_flag
1,C00001,rushjeff@ryan.org,Johnsonmouth,MS,Emily Mooney,ryan.org,Other,2026-09-10T12:45:41.310Z,2026-09-10T12:45:41.310Z,true
2,C00002,mccoykiara@kelly.com,Stephenfort,WY,Andrea Sellers,kelly.com,Other,2026-09-10T12:45:41.310Z,2026-09-10T12:45:41.310Z,true
3,C00003,rebeccamiller@yahoo.com,South Stephenshire,LA,Craig Hayes,yahoo.com,Yahoo,2026-09-10T12:45:41.310Z,2026-09-10T12:45:41.310Z,true
4,C00004,lawrence05@campbell.info,Chrisland,ND,Bryan Scott,campbell.info,Other,2026-09-10T12:45:41.310Z,2026-09-10T12:45:41.310Z,true
5,C00005,carrie45@yahoo.com,East Dennistown,RI,Sean Vasquez,yahoo.com,Yahoo,2026-09-10T12:45:41.310Z,2026-09-10T12:45:41.310Z,true
6,C00006,traceyramos@gmail.com,North Matthew,IN,Kevin Mccarthy,gmail.com,Gmail,2026-09-10T12:45:41.310Z,2026-09-10T12:45:41.310Z,true
7,C00007,scottallen@gmail.com,Joneshaven,VA,Amanda Doyle,gmail.com,Gmail,2026-09-10T12:45:41.310Z,2026-09-10T12:45:41.310Z,true
8,C00008,sullivanjeremy@horton-adams.com,South Nathanfurt,CT,Paul Campos,horton-adams.com,Other,2026-09-10T12:45:41.310Z,2026-09-10T12:45:41.310Z,true
9,C00009,dennis03@yahoo.com,Kimberlyview,MD,Mary Green,yahoo.com,Yahoo,2026-09-10T12:45:41.310Z,2026-09-10T12:45:41.310Z,true
10,C00010,charles58@murillo.net,West Hector,OK,James Myers,murillo.net,Other,2026-09-10T12:45:41.310Z,2026-09-10T12:45:41.310Z,true
